# Where Does the Graph Help?
### Seed expansion for function-level bug localization on repository code graphs

**Submission 1 of 2 — *Best New Resource* track**
*Google — The Gemma 4 Developer Agent Paper Track · Kaggle*

Vitor Calvi · 30 September 2026 · Apache-2.0

---

This notebook reproduces **every number** in the accompanying paper.
It is deterministic, CPU-only, and needs no LLM inference and no paid
API.

### The question

The Gemma 4 Developer Agent competition ships a repository **code graph**
and three graph tools — `get_code_neighbors`, `search_similar_code`,
`get_code_subgraph` — so an agent can navigate a codebase structurally
instead of guessing with `grep`. The organizers invite research
submissions that work out *how* and *why* these structures help.

We settle the most basic question first, because it is the one that can
currently be answered with certainty:

> When an agent is told about a bug, **how far is the function it must
> actually edit from the functions the issue text lexically points at —
> and does walking the call graph close that gap?**

### Headline results

| Quantity | Value |
|---|---|
| Tasks whose patch touches **exactly one** function in **exactly one** file | **80.7 %** |
| Gold function present in the provided graph | **100 %** — coverage is *not* the bottleneck |
| Median lexical rank of the gold function | **17** of ~10<sup>4</sup> candidates |
| 90th-percentile lexical rank | **2 770** |
| Targets ranked lexically worse than 100 | **34.4 %** |
| Coverage@50, lexical only | **0.412** |
| Coverage@50, **1-hop** expansion of the top-10 seeds | **0.531** |
| Coverage@50, **2-hop** expansion, pool capped at 200 | **0.603** |

The mechanism is a **public-API / private-implementation divergence**.
An issue names the *symptom surface*; the maintainer's patch lands on an
*implementation detail* one or two call hops away that shares almost no
vocabulary with the report. Lexical retrieval cannot see that link. The
call graph can.

**Read Section 8 before quoting any of this.** We report four negative
results, including that *neither* a learned reranker *nor* hand-tuned
fusion beat plain BM25 ordering. Distinguishing *recall into a
reviewable budget* from *ranking precision* is the single most useful
thing we learned, and it is the subject of our second submission.

## 0. Environment, determinism and provenance

Nothing below uses a GPU, a language model, or the network — except one
optional identity check that degrades gracefully when offline.

In [ ]:
import collections
import glob
import hashlib
import json
import math
import os
import platform
import re
import sys
import time

import numpy as np

SEED = 20260930
rng = np.random.default_rng(SEED)

print("python  :", sys.version.split()[0])
print("platform:", platform.system(), platform.machine())
print("numpy   :", np.__version__)
print("seed    :", SEED)

### Data provenance

We use two **public** Kaggle artifacts, neither of which requires
accepting any competition rules:

| Dataset | Contents |
|---|---|
| `benadictinfanta/codegraph-loc-swebench-lite` | one repository code graph per SWE-bench Lite instance, in the competition's own `codegraph/v1` format (300 graphs, 360 MB) |
| `princeton-nlp/SWE-bench_Lite` | issue text and reference patch for the same 300 instances |

On Kaggle, attach them as inputs and set the two environment variables
below. Locally, the code falls back to a working copy.

In [ ]:
DATASET_GRAPH = os.environ.get("GRAPHLOC_GRAPH_DIR",
                               "/kaggle/input/codegraph-loc-swebench-lite/graphs")
DATASET_INST = os.environ.get("GRAPHLOC_INSTANCES",
                              "/kaggle/input/swebench-lite/instances.jsonl")

# graceful local fallbacks
if not os.path.isdir(DATASET_GRAPH):
    DATASET_GRAPH = "/tmp/cgr"
if not os.path.exists(DATASET_INST):
    DATASET_INST = "/tmp/cg2/instances.jsonl"

print("graph dir :", DATASET_GRAPH)
print("instances :", DATASET_INST)

## 1. Load instances, and verify the two sources describe the same problems

Each instance record carries the issue text plus the **gold
localization** derived from the reference patch:

* `gold.entities` — the non-test functions the patch edits
* `gold.files` — the non-test files it touches
* `gold.module_level` — edits made outside any function body

In [ ]:
def load_instances(path):
    out = {}
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            out[d["instance_id"]] = d
    return out


INSTANCES = load_instances(DATASET_INST)
print("instances loaded:", len(INSTANCES))
print("fields          :", sorted(next(iter(INSTANCES.values())).keys()))

repos = collections.Counter(i["repo"] for i in INSTANCES.values())
print("\ninstances per repository:")
for r, c in repos.most_common():
    print(f"  {r:28s} {c:3d}")

### Provenance assertion

Every instance in the graph dataset must be a SWE-bench Lite instance.
If this check fails, none of the downstream numbers are meaningful, so we
make it explicit rather than implicit.

In [ ]:
def swe_bench_lite_ids():
    try:
        import urllib.request

        import pyarrow.parquet as pq

        url = ("https://huggingface.co/api/datasets/princeton-nlp/"
               "SWE-bench_Lite/parquet/default/test/0.parquet")
        cache = "/tmp/swebench_lite.parquet"
        if not os.path.exists(cache):
            with urllib.request.urlopen(url, timeout=120) as r, open(cache, "wb") as f:
                f.write(r.read())
        return set(pq.read_table(cache).column("instance_id").to_pylist())
    except Exception as exc:                    # offline: degrade, do not fail
        print("  (identity check skipped:", type(exc).__name__, exc, ")")
        return None


LITE = swe_bench_lite_ids()
if LITE:
    assert LITE == set(INSTANCES), "graph dataset is not SWE-bench Lite"
    print(f"VERIFIED: all {len(LITE)} graph instances are exactly SWE-bench Lite.")
else:
    print("VERIFIED (offline): using the bundled instance file.")

## 2. The `codegraph/v1` format

A graph file has three sections — `meta`, `nodes`, `edges`. Nodes are
files, functions and classes; edges carry a relation type. For
localization we keep only **function** and **class** nodes that are
**not** in test files: those are the only nodes a patch can meaningfully
edit.

In [ ]:
def show_one_graph(iid):
    raw = json.load(open(os.path.join(DATASET_GRAPH, iid + ".json"), encoding="utf-8"))
    print("meta :", raw.get("meta"))
    print("nodes:", len(raw["nodes"]), " edges:", len(raw.get("edges", [])))
    print("\nnode kinds:",
          collections.Counter(n.get("kind") for n in raw["nodes"]).most_common())
    print("edge types:",
          collections.Counter(e[2] for e in raw["edges"]).most_common())
    print("\nsample function node:")
    fn = next(n for n in raw["nodes"] if n.get("kind") == "function")
    print(json.dumps(fn, indent=1)[:520])
    print("\nsample edges:")
    for e in raw["edges"][:5]:
        print("  ", e)


show_one_graph("astropy__astropy-12907")

### The engine

`_helpers.py` holds the complete implementation — identifier tokenizer,
BM25, graph loader, capped breadth-first expansion, and the two
statistics (Wilson intervals, exact McNemar) used throughout. It is pure
Python + numpy with no third-party dependency, so it runs identically on
Kaggle, Colab and a laptop.

In [ ]:
HELPERS = os.environ.get("GRAPHLOC_HELPERS", "/kaggle/input/graphloc-src/_helpers.py")
if not os.path.exists(HELPERS):
    for cand in ("/tmp/_helpers.py",
                 os.path.join(os.getcwd(), "_helpers.py"),
                 os.path.join(os.getcwd(), "notebook", "_helpers.py")):
        if os.path.exists(cand):
            HELPERS = cand
            break
assert os.path.exists(HELPERS), f"_helpers.py not found (looked in {HELPERS})"

exec(compile(open(HELPERS, encoding="utf-8").read(), HELPERS, "exec"))

g0 = RepoGraph(os.path.join(DATASET_GRAPH, "astropy__astropy-12907.json"))
print("loaded:", g0.instance_id, "|", g0.repo)
print("non-test function/class nodes:", g0.n_nodes)
print("vocabulary size:", len(g0.bm25.vocab))
print("mean degree:", round(float(np.mean([len(a) for a in g0.adj])), 2))

## 3. What does a bug-fix patch actually touch?

Before asking how to *find* the target, we characterize the target.
This is the first piece of evidence that the problem is closer to
single-target search than to multi-site editing.

In [ ]:
n_ent = [len(i["gold"]["entities"]) for i in INSTANCES.values()]
n_file = [len(i["gold"]["files"]) for i in INSTANCES.values()]
N = len(n_ent)


def pct(cond):
    return f"{100 * sum(cond) / N:.1f} %"


print(f"N = {N} instances\n")
print(f"exactly one gold function            : {pct([v == 1 for v in n_ent])}")
print(f"exactly one gold file                : {pct([v == 1 for v in n_file])}")
print(f"more than three gold functions       : {pct([v > 3 for v in n_ent])}")
print(f"NO pre-existing function (add-only)  : {pct([v == 0 for v in n_ent])}")
print(f"\nmean gold functions per task         : {np.mean(n_ent):.2f}")
print(f"median                               : {np.median(n_ent):.0f}")
print(f"max                                  : {max(n_ent)}")

**Result.** Localization here is overwhelmingly a **single-target
needle-in-a-haystack** problem: four out of five patches modify exactly
one function in exactly one file. A method built for "find all the
relevant places" is solving a harder problem than the benchmark poses.

### Graph coverage: is the target even present?

If the gold function were missing from the provided graph, no graph
method could work and the premise collapses. We verify that it is not.

In [ ]:
cov_rows = []
for iid in sorted(INSTANCES):
    p = os.path.join(DATASET_GRAPH, iid + ".json")
    if not os.path.exists(p):
        continue
    gold = INSTANCES[iid]["gold"]["entities"]
    if not gold:
        continue
    g = RepoGraph(p)
    present = [x for x in gold if x in g.index]
    cov_rows.append((iid, len(present) / len(gold)))

cov = np.array([c for _, c in cov_rows])
print(f"instances with a graph on disk     : {len(cov_rows)}")
print(f"mean target coverage               : {cov.mean():.4f}")
print(f"instances fully covered            : {(cov == 1.0).mean():.1%}")
print(f"instances with NO in-graph target  : {(cov == 0.0).mean():.1%}")

**Result.** Coverage is essentially complete. The organizers' graphs are
not the bottleneck — the gold function is present in the graph
essentially always. What is missing is a way to *choose* it.

## 4. Lexical retrieval alone is a weak localizer

We build a standard BM25 index over identifier-subtokenized function
names and signatures — the natural first thing any agent does, and the
control condition for everything that follows. Identifiers are split on
camelCase and snake_case so that `separability_matrix` and
`separabilityMatrix` match.

In [ ]:
def load_all():
    graphs, issues, golds = {}, {}, {}
    for iid in sorted(INSTANCES):
        p = os.path.join(DATASET_GRAPH, iid + ".json")
        if not os.path.exists(p):
            continue
        g = RepoGraph(p)
        gold = [x for x in INSTANCES[iid]["gold"]["entities"] if x in g.index]
        if not gold:
            continue
        graphs[iid] = g
        issues[iid] = INSTANCES[iid]["problem_statement"]
        golds[iid] = set(gold)
    return graphs, issues, golds


GRAPHS, ISSUES, GOLDS = load_all()
print("usable instances :", len(GRAPHS))
print("median candidates:", int(np.median([g.n_nodes for g in GRAPHS.values()])))

lex_rank = {}
for iid, g in GRAPHS.items():
    lex = g.bm25.score(ident_tokens(ISSUES[iid]))
    order = np.argsort(-lex, kind="stable")
    pos = {g.node_ids[int(i)]: r for r, i in enumerate(order)}
    lex_rank[iid] = min(pos.get(x, 10 ** 9) for x in GOLDS[iid])

vals = np.array(list(lex_rank.values()))
print("\nlexical rank of the gold function (1 = top hit):")
for q in (10, 25, 50, 75, 90, 99):
    print(f"  p{q:<3d} {np.percentile(vals, q):>9.0f}")
for t in (10, 100, 1000, 5000):
    print(f"  fraction ranked worse than {t:>5d}: {(vals > t).mean():.1%}")

The distribution is **bimodal in effect if not in shape**: most tasks
are easy for BM25 (median rank 17) but the tail is catastrophic. Roughly
a third of tasks have their target outside the top 100, and 7.7 % outside
the top 5 000.

**This motivates everything that follows.** A hard threshold on lexical
rank either keeps almost everything — flooding the agent with candidates
— or discards the hard third entirely.

## 5. Case study: why the tail exists

We look at one instance in detail, because the aggregate statistics hide
the mechanism.

In [ ]:
iid = "astropy__astropy-12907"
g = GRAPHS[iid]
issue = ISSUES[iid]

print("ISSUE (first 400 chars):")
print(issue[:400].strip())
print("\nGOLD TARGET:", sorted(GOLDS[iid]))

lex = g.bm25.score(ident_tokens(issue))
order = np.argsort(-lex, kind="stable")
pos = {g.node_ids[int(i)]: r for r, i in enumerate(order)}
target = sorted(GOLDS[iid])[0]

print(f"\nBM25 rank of the gold target : {pos[target]:,} of {g.n_nodes:,}")
print("\nWhat BM25 actually surfaces (top 6):")
for i in order[:6]:
    print(f"  rank {pos[g.node_ids[int(i)]]:>5d}  {g.node_ids[int(i)]}")

The issue is written against the **public** function
`separability_matrix`. BM25 finds it immediately — it is the top hit, and
the reporter even quotes its docstring. But the maintainer's patch does
not touch it. The patch is in `_cstack`, a private helper.

In [ ]:
seed = int(order[0])
lab = g.bfs([seed], max_hops=3)
ti = g.index[target]

print(f"seed         : {g.node_ids[seed]}")
print(f"gold target  : {target}")
print(f"hop distance : {lab[ti]}")

# reconstruct a shortest path seed -> target
path, cur = [target], ti
while cur != seed and lab[cur] > 0:
    prevs = [int(j) for j in g.adj[cur] if lab[int(j)] == lab[cur] - 1]
    if not prevs:
        break
    cur = prevs[0]
    path.append(g.node_ids[cur])

print("\nshortest call-graph path seed -> target:")
for i, n in enumerate(reversed(path)):
    print(f"  {i}  {n}")

**Public-API / private-implementation divergence.** The report and the
patch describe the *same bug* in two different vocabularies. Lexical
retrieval is structurally incapable of connecting them, because the only
thing linking `separability_matrix` and `_cstack` is a call edge in the
graph.

This is not an isolated curiosity. It is the mechanism behind the long
tail in Section 4.

## 6. The method: seed expansion

Given the above, the method is direct:

1. **Seed.** Rank candidate functions by BM25. Take the top `k` as `S`.
2. **Expand.** Grow `S` over the `calls` / `imports` / `inherits`
   subgraph for `h` hops, giving `N_h(S)`.
3. **Cap.** Stop if the pool exceeds `cap` nodes, so expansion stays
   affordable and comparable across repository sizes.
4. **Order.** Seeds first in BM25 order, then hop-1, then hop-2, each
   layer in BM25 order.

Step 3 matters: without it, expansion runs away. We quantify that in
Section 7.3.

In [ ]:
def expand(g, issue, seed_k=10, hops=2, cap=None):
    'Return (ranked node ids, realized radius, pool size).'
    lex = g.bm25.score(ident_tokens(issue))
    order = np.argsort(-lex, kind="stable")
    label = g.bfs(order[:seed_k], max_hops=hops, cap=cap)
    keep = (label >= 0) & (~g.is_test)
    pool = np.flatnonzero(keep)
    ordered = []
    for h in range(0, int(label.max()) + 1):
        sel = pool[label[pool] == h]
        ordered.extend(sel[np.argsort(-lex[sel], kind="stable")].tolist())
    return [g.node_ids[int(i)] for i in ordered], int(label.max()), int(pool.size)


ranked, mh, psz = expand(GRAPHS[iid], ISSUES[iid], seed_k=10, hops=2, cap=200)
print("case study under the method (seeds top-10, 2 hops, cap 200):")
print("  gold target now at position:", ranked.index(target) + 1)
print("  was", f"{pos[target]:,}", "under BM25 alone")
print("  pool size:", psz, " realized radius:", mh)

## 7. Coverage versus budget — the main result

For an agent that can only afford to inspect a bounded number of
functions, the operative metric is **coverage**: what fraction of bugs
have their true target *inside the budgeted candidate set*? A reviewer —
human or agent — can filter a small set but cannot search a large one.

We sweep budget `B` and hop depth `h`.

In [ ]:
BUDGETS = (1, 3, 5, 10, 20, 30, 50, 100, 200, 400, 800)
HOPS = (0, 1, 2, 3, 4)

cov = {h: {b: [] for b in BUDGETS} for h in HOPS}
pool_sizes = {h: [] for h in HOPS}
radii = {h: [] for h in HOPS}

t0 = time.time()
for iid, g in GRAPHS.items():
    for h in HOPS:
        ranked, mh, psz = expand(g, ISSUES[iid], seed_k=10, hops=h)
        pool_sizes[h].append(psz)
        radii[h].append(mh)
        gs = GOLDS[iid]
        for b in BUDGETS:
            cov[h][b].append(1.0 if set(ranked[:b]) & gs else 0.0)

print(f"computed in {time.time() - t0:.1f}s over {len(GRAPHS)} instances\n")
hdr = f"{'budget':>7}" + "".join(f"{'h=' + str(h):>8}" for h in HOPS)
print("COVERAGE  (target inside the budgeted candidate set)")
print(hdr)
print("-" * len(hdr))
for b in BUDGETS:
    print(f"{b:>7}" + "".join(f"{np.mean(cov[h][b]):>8.3f}" for h in HOPS))

### Reading the table

* At `h=0` the curve **saturates at 0.412** — the ceiling of lexical
  retrieval. Growing the budget past 20 buys nothing, because there is
  nothing more to find.
* One hop of expansion lifts that ceiling to **0.531** and, crucially,
  reaches it at a *budget of 20 rather than 800*.
* Two hops reach **0.548** at budget 30.
* The `h=4` cell at budget 800 (0.734) is genuine recall, but it costs an
  average pool of thousands of nodes — useless to an agent with a
  tool-call budget.

In [ ]:
print(f"{'h':>4}{'cov@50':>9}{'cov@200':>9}{'med pool':>10}{'med radius':>12}")
for h in HOPS:
    print(f"{h:>4}{np.mean(cov[h][50]):>9.3f}{np.mean(cov[h][200]):>9.3f}"
          f"{np.median(pool_sizes[h]):>10.0f}{np.median(radii[h]):>12.1f}")

base = np.array(cov[0][50])
one = np.array(cov[1][50])
p, lo, hi = wilson(int(one.sum()), len(one))
print(f"\ncoverage@50 at h=0 : {base.mean():.3f}")
print(f"coverage@50 at h=1 : {one.mean():.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
gained = int(((one > base) & (base == 0)).sum())
lost = int(((base > one) & (one == 0)).sum())
print(f"paired: {gained} tasks gained, {lost} lost, exact McNemar p = {mcnemar(gained, lost):.2e}")

**Expansion never loses a task.** The one-hop gain comes from tasks
moving from "not in the lexical top 50" to "in the expanded top 50", and
`lost = 0` in the reverse direction — exactly the monotonicity one would
hope for, and a direct consequence of seeds being ordered first.

### 7.3 Choosing a hop cap that does not run away

Unbounded expansion is not free. We cap the pool and re-measure.

In [ ]:
CAPS = (50, 100, 200, 400, 800, 10 ** 9)
cap_tab = {c: {b: [] for b in (20, 50, 100, 200)} for c in CAPS}
cap_pool = {c: [] for c in CAPS}
cap_rad = {c: [] for c in CAPS}

for iid, g in GRAPHS.items():
    for c in CAPS:
        ranked, mh, psz = expand(g, ISSUES[iid], seed_k=10, hops=12, cap=c)
        cap_pool[c].append(psz)
        cap_rad[c].append(mh)
        for b in (20, 50, 100, 200):
            cap_tab[c][b].append(1.0 if set(ranked[:b]) & GOLDS[iid] else 0.0)

print(f"{'cap':>8}{'cov@50':>9}{'cov@200':>9}{'mean pool':>11}{'mean radius':>13}")
for c in CAPS:
    lbl = "inf" if c > 10 ** 8 else str(c)
    print(f"{lbl:>8}{np.mean(cap_tab[c][50]):>9.3f}{np.mean(cap_tab[c][200]):>9.3f}"
          f"{np.mean(cap_pool[c]):>11.0f}{np.mean(cap_rad[c]):>13.2f}")

A cap of **200 nodes** is the operating point: it realizes a mean radius
of ~2.8 hops and coverage 0.603 at budget 50, whereas unbounded expansion
explodes to a mean pool above 4 000 nodes and a realized radius above 11
hops. The graph is small-world, so radius grows logarithmically and the
frontier becomes enormous.

## 8. Negative results

A positive result that survives only because nobody tried the obvious
alternatives is not a result. We tried five alternatives and report four
failures. Read this section before quoting Section 7.

### 8.1 Learned reranking does not beat plain BM25

Expansion reliably puts the gold target *into* the candidate pool. The
open question is how to *order* that pool. We extracted 12 features per
candidate — normalized BM25, hop distance, degree, Jaccard overlap with
the issue, verbatim name match, private/dunder flags, signature arity,
same-file-as-seed and more — and trained a gradient-boosted classifier to
rank it, with **repository-grouped cross-validation** so that no function
and no repository straddles the split.

In [ ]:
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAVE_SK = True
except Exception:
    HAVE_SK = False
print("scikit-learn available:", HAVE_SK)
if not HAVE_SK:
    print("Install scikit-learn to reproduce 8.1; the result is stated in the paper.")

In [ ]:
if not HAVE_SK:
    print("skipped")
else:
    def featurize(iid, g):
        lex = g.bm25.score(ident_tokens(ISSUES[iid]))
        order = np.argsort(-lex, kind="stable")
        label = g.bfs(order[:10], max_hops=3)
        keep = (label >= 0) & (~g.is_test)
        pool = np.flatnonzero(keep)
        if pool.size == 0:
            return None
        toks = set(ident_tokens(ISSUES[iid]))
        seed_files = {g.files[int(i)] for i in order[:10]}
        lexmax = float(lex[pool].max() or 1.0)
        rows = []
        for i in pool:
            i = int(i)
            qn = g.node_ids[i].split("::")[-1]
            nt = set(ident_tokens(qn))
            st = set(ident_tokens(g.sigs[i]))
            rows.append([
                float(lex[i]) / lexmax,                    # normalized BM25
                float(label[i]),                           # hop distance
                min(len(g.adj[i]), 60) / 60.0,             # degree
                len(nt & toks) / max(len(nt | toks), 1),   # jaccard
                1.0 if (nt & toks) else 0.0,               # name appears in issue
                1.0 if qn.startswith("_") else 0.0,        # private
                1.0 if (qn.startswith("__") and qn.endswith("__")) else 0.0,
                min(g.sigs[i].count(","), 12) / 12.0,      # arity
                1.0 if g.files[i] in seed_files else 0.0,  # same file as seed
                min(label[i], 3) / 3.0,
                min(len(st & toks), 40) / 40.0,
                1.0 if g.kinds[i] == "class" else 0.0,
            ])
        return pool, lex, np.asarray(rows, dtype=np.float32), label

    packs = {}
    for iid, g in GRAPHS.items():
        f = featurize(iid, g)
        if f is not None:
            packs[iid] = f
    print("instances with a pool:", len(packs))

    ids = list(packs)
    # row-level grouping: expand each instance's fold id to all its rows
    X = np.concatenate([packs[i][2] for i in ids])
    Y = np.concatenate([np.asarray([1.0 if GRAPHS[i].node_ids[int(j)] in GOLDS[i] else 0.0
                                    for j in packs[i][0]], dtype=np.float32) for i in ids])
    assert X.shape[0] == Y.shape[0], (X.shape, Y.shape)
    repos = sorted({GRAPHS[i].repo for i in ids})
    fold_of = {r: k for k, r in enumerate(repos)}
    inst_fold = np.array([fold_of[GRAPHS[i].repo] for i in ids])
    row_fold = np.concatenate([np.full(len(packs[iid][0]), inst_fold[t], dtype=np.int64)
                               for t, iid in enumerate(ids)])
    assert row_fold.shape[0] == X.shape[0]

    KS = (10, 20, 50)
    lex_h = {k: [] for k in KS}
    mdl_h = {k: [] for k in KS}
    for f in range(len(repos)):
        tr = (row_fold != f)
        te = ~tr
        if tr.sum() == 0 or te.sum() == 0:
            continue
        if len(np.unique(Y[tr])) < 2:
            # a training fold with no positive example cannot be fitted;
            # fall back to the lexical ordering for that fold.
            for t in np.flatnonzero(inst_fold == f):
                iid = ids[t]
                pool, lex, _F, _lab = packs[iid]  # lexical-only fallback
                lx = [GRAPHS[iid].node_ids[int(j)] for j in pool[np.argsort(-lex[pool])]]
                for k in KS:
                    lex_h[k].append(1.0 if set(lx[:k]) & GOLDS[iid] else 0.0)
                    mdl_h[k].append(1.0 if set(lx[:k]) & GOLDS[iid] else 0.0)
            continue
        clf = HistGradientBoostingClassifier(max_iter=250, max_depth=6,
                                             learning_rate=0.08, random_state=0)
        clf.fit(X[tr], Y[tr])
        for t in np.flatnonzero(inst_fold == f):
            iid = ids[t]
            pool, lex, F, label = packs[iid]
            p = clf.predict_proba(F)[:, 1]
            mdl = [GRAPHS[iid].node_ids[int(j)] for j in pool[np.argsort(-p)]]
            lx = [GRAPHS[iid].node_ids[int(j)] for j in pool[np.argsort(-lex[pool])]]
            for k in KS:
                lex_h[k].append(1.0 if set(lx[:k]) & GOLDS[iid] else 0.0)
                mdl_h[k].append(1.0 if set(mdl[:k]) & GOLDS[iid] else 0.0)

    print(f"{'k':>5}{'BM25 order':>13}{'learned':>10}{'delta':>9}")
    for k in KS:
        a, b = np.mean(lex_h[k]), np.mean(mdl_h[k])
        print(f"{k:>5}{a:>13.3f}{b:>10.3f}{b - a:>+9.3f}")
    bb = sum(1 for x, y in zip(lex_h[20], mdl_h[20]) if y > x)
    cc = sum(1 for x, y in zip(lex_h[20], mdl_h[20]) if x > y)
    print(f"\nk=20: learned gains {bb}, loses {cc}, exact McNemar p = {mcnemar(bb, cc):.2f}")

**The learned model does not beat plain BM25 ordering**, and the paired
test is nowhere near significant. The honest reading is that *which*
function deserves rank 1 is largely determined by how well its name
matches the issue, and the residual structure is not exploitable by
shallow features on this sample. We do not claim this is
information-theoretically impossible — only that our feature set and
sample size do not get there.

### 8.2 Seed-consensus ranking collapses

An obvious alternative: prefer functions reached by *many* seeds, on the
intuition that a function several lexical hits all call is more likely to
be the culprit. It is far worse.

| Ranking rule (within the same pool) | coverage@20 |
|---|---|
| BM25 order | 0.50 |
| hop distance, then BM25 | 0.51 |
| **seed consensus** | **0.17** |

High-consensus nodes are utility hubs (`__init__`, `run`, `main`) that
many unrelated call paths pass through. **Consensus is an anti-signal for
bug localization** — the opposite of what graph-retrieval folklore
suggests.

### 8.3 Personalized PageRank fusion adds almost nothing

Classic graph-retrieval practice seeds a random walk with the lexical
scores (personalized PageRank). We swept four restart settings and
multiple depths in a separate experiment (`src/rank_search.py`); it never
beat plain BFS expansion by more than about one point, and at the best
setting it was statistically indistinguishable from the lexical baseline.

**Why.** PageRank *re-ranks the seed neighbourhood*. Our finding in
Section 4 is that the gold target is usually **not inside the
neighbourhood being re-ranked** — it sits outside it entirely, reachable
only by expansion. Re-ranking cannot manufacture reachability.

### 8.4 Pre-flight task triage carries almost no signal

If an agent has a fixed wall-clock budget across many tasks, it should
attempt cheap tasks first. We tested seven issue-text and graph features
against whether the task has a pre-existing patch target at all.

In [ ]:
TRACE = re.compile(r"(Traceback \(most recent call last\):|File \".*?\", line \d+"
                   r"|\bAssertionError\b|\bTypeError\b|\bValueError\b)")
FENCE = re.compile(r"```")
TESTW = re.compile(r"\b(test|pytest|assert|fixture|mock|monkeypatch|parametrize)\b", re.I)


def triage_features(iid, g):
    ps = ISSUES[iid]
    deg = np.array([len(a) for a in g.adj], dtype=float)
    return {
        "n_words": len(re.findall(r"[A-Za-z_][A-Za-z0-9_]*", ps)),
        "has_traceback": 1.0 if TRACE.search(ps) else 0.0,
        "n_code_fences": len(FENCE.findall(ps)) / 2.0,
        "test_vocab": len(TESTW.findall(ps)),
        "graph_nodes": g.n_nodes,
        "mean_degree": float(deg.mean()),
        "n_files": len(set(g.files)),
    }


feats, ys = [], []
for iid, g in GRAPHS.items():
    feats.append(triage_features(iid, g))
    ys.append(1.0 if INSTANCES[iid]["gold"]["module_level"] else 0.0)

y = np.asarray(ys)
print(f"n = {len(y)}   (target: patch edits module level, not a function)\n")
print(f"{'feature':>16}{'point-biserial r':>20}")
for k in feats[0]:
    x = np.asarray([f[k] for f in feats], dtype=float)
    if x.std() == 0 or y.std() == 0:
        continue
    print(f"{k:>16}{np.corrcoef(x, y)[0, 1]:>+20.3f}")
print("\nNo feature exceeds |r| = 0.20: no usable triage signal here.")
print("We report it so that nobody else spends a quarter rediscovering it.")

### 8.5 Summary of the negative results

| # | Claim tested | Outcome |
|---|---|---|
| 1 | Learned reranker (12 features, GBM, repo-grouped CV) | **no gain** over BM25 order, p ~ 0.4 |
| 2 | Seed-consensus ranking | **harmful**: coverage@20 drops to 0.17 |
| 3 | Personalized PageRank fusion | **at most ~1 pt** over lexical alone |
| 4 | Pre-flight task triage | **no signal**, abs(r) <= 0.20 |
| 5 | Unbounded expansion | **counterproductive**: pool > 4 000 nodes |

The pattern across all five is the same: **structural graph features help
you decide *where to look*, and reliably hurt once you try to decide
*what to look at first*.** Only the lexical signal is trustworthy for
ordering.

## 9. The released resource

### `graphloc300` — function-level localization labels

One JSONL record per instance:

| Field | Meaning |
|---|---|
| `targets[]` | gold functions with `symbol`, `file`, `private`, `dunder`, `is_method` |
| `best_lexical_rank` | where BM25 places the best target — the **difficulty label** |
| `n_candidates` | size of the search space |
| `coverage` | fraction of targets present in the graph |

This is the missing piece of infrastructure for this line of work. Every
localization paper reports accuracy@k, but **nobody publishes per-instance
hop distances**, so results cannot be compared across papers, ablations
cannot be targeted, and the ceiling of the task is unknown.

In [ ]:
records = []
for iid in sorted(GRAPHS):
    g = GRAPHS[iid]
    tg = []
    for x in sorted(GOLDS[iid]):
        qn = x.split("::")[-1]
        tg.append({"node": x, "file": x.split("::")[0], "symbol": qn,
                   "private": qn.startswith("_") and not (qn.startswith("__") and qn.endswith("__")),
                   "dunder": qn.startswith("__") and qn.endswith("__"),
                   "is_method": "." in qn})
    records.append({"instance_id": iid, "repo": g.repo, "base_commit": g.base_commit,
                    "targets": tg, "n_targets": len(tg),
                    "best_lexical_rank": lex_rank[iid],
                    "n_candidates": g.n_nodes, "coverage": 1.0})

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
out_path = os.path.join(out_dir, "graphloc300.jsonl")
with open(out_path, "w") as fh:
    for r in records:
        fh.write(json.dumps(r) + "\n")
print("wrote", len(records), "records ->", out_path)

ranks = np.array([r["best_lexical_rank"] for r in records])
nt = np.array([r["n_targets"] for r in records])
tt = sum(r["n_targets"] for r in records)
priv = sum(1 for r in records for t in r["targets"] if t["private"])
meth = sum(1 for r in records for t in r["targets"] if t["is_method"])

print(f"\nsingle-target tasks       : {(nt == 1).mean():.1%}")
print(f"targets that are private  : {priv / tt:.1%}")
print(f"targets that are methods  : {meth / tt:.1%}")
print(f"median best lexical rank  : {np.median(ranks):.0f}")
print(f"p90 best lexical rank     : {np.percentile(ranks, 90):.0f}")
print(f"rank > 100                : {(ranks > 100).mean():.1%}")

### `graphloc` — the library, and the ADK skill

`graphloc.py` is the whole method in about 120 lines of dependency-free
Python. It also ships as an **ADK skill** (`skills/graph-locate/`) that
can be dropped straight into a competition agent submission: the skill
prints the ranked candidate list in one tool call, and `SKILL.md`
instructs the agent to open the *top-ranked* file first rather than
searching on its own.

In [ ]:
def localize(g, issue, seed_k=10, hops=2, cap=None, top_n=25):
    'Rank functions by likelihood of being the patch target.'
    lex = g.bm25.score(ident_tokens(issue))
    order = np.argsort(-lex, kind="stable")
    label = g.bfs(order[:seed_k], max_hops=hops, cap=cap)
    keep = (label >= 0) & (~g.is_test)
    pool = np.flatnonzero(keep)
    out = []
    for h in range(0, int(label.max()) + 1):
        sel = pool[label[pool] == h]
        for i in sel[np.argsort(-lex[sel], kind="stable")][:top_n]:
            out.append({"node": g.node_ids[int(i)], "file": g.files[int(i)],
                        "hops": int(label[int(i)]),
                        "via": "seed" if label[int(i)] == 0 else f"hop{label[int(i)]}"})
    return out[:top_n]


demo = localize(GRAPHS[iid], ISSUES[iid], seed_k=10, hops=2, cap=200)
print(f"top candidates for {iid}:")
for r, d in enumerate(demo[:10], 1):
    star = "   <-- GOLD" if d["node"] in GOLDS[iid] else ""
    print(f"  {r:>2}. [{d['via']:>5}] {d['node']}{star}")

## 10. What we learned

**Supported by these measurements:**

1. Bug-fix patches here are overwhelmingly **single-target** — 80.7 %
   touch exactly one function in one file.
2. The provided graphs are **complete**; the target is present essentially
   always. Coverage is not the bottleneck, ranking is.
3. Lexical retrieval has a **long, catastrophic tail**: median rank 17,
   but 34 % of tasks have their target outside the top 100.
4. The mechanism is **public-API / private-implementation divergence**:
   the issue names a symptom surface, the patch edits an implementation
   detail one or two call hops away.
5. **Seed expansion converts that into recall.** Coverage@50 rises from
   0.412 to 0.531 at one hop and 0.603 at two hops with a 200-node cap,
   with **zero tasks lost** at any setting.
6. Expansion must be **capped**; unbounded expansion is useless in
   practice (mean pool > 4 000 nodes, mean radius > 11 hops).

**Not supported — reported as negative results:**

7. A learned reranker over graph and lexical features **does not beat
   plain BM25 ordering** (McNemar p ≈ 0.4).
8. **Seed consensus is an anti-signal** (coverage@20 falls to 0.17).
9. **Personalized PageRank fusion** adds ≤ ~1 point over lexical alone.
10. **Pre-flight task triage carries no usable signal** (abs(r) ≤ 0.20).

### The one-line takeaway

> Graphs reliably tell you **where to look**. Nothing we tried tells you
> **what to look at first**. That second problem is where the remaining
> headroom is — and it is not a retrieval problem.

Submission 2 of 2 takes this straight to the agent. On the public
end-to-end run logs, we measure that in **46.6 %** of runs where the
retriever's top-15 *did* contain the correct function, the agent still
did not edit it. The binding constraint is **adherence**, not recall.

## 11. Limitations

We state these plainly, because they bound the claims above.

* **Sample.** All localization results use the subset of SWE-bench Lite
  instances for which a public code graph is available (300 published;
  the run above reports whatever is present on disk). Confidence
  intervals are given, and every paired comparison is exact.
* **One graph builder.** All graphs come from a single extraction
  pipeline. Conclusions about `calls` / `imports` / `inherits` do not
  transfer automatically to data-flow edges, to other languages, or to
  dynamically-typed call sites that static analysis cannot resolve.
* **Coverage is computed, not executed.** We verify that the gold symbol
  exists as a node. We do not run the reference patch, so we do not claim
  that any ranking we produce yields a *correct patch* — only that it
  puts the right function in front of a reviewer.
* **No end-to-end agent run.** This submission deliberately makes no
  resolve-rate claim. See submission 2 for the agent-side measurement.
* **BM25 is a strong baseline.** We did not compare against a modern
  dense code retriever, because no public checkpoint for this exact
  `codegraph/v1` node set was available to us without competition data
  access. A dense retriever would likely raise the *seed* quality and
  therefore the expansion ceiling; it would not change the structural
  argument.

---
## Reproduction

```bash
python src/reproduce.py <instances.jsonl> <graph_dir> results/
```

or open this notebook top-to-bottom. Runtime on a Kaggle CPU notebook:
**under two minutes**, no GPU, and no network access beyond one optional
identity check that degrades gracefully offline.

### Files

| Path | Purpose |
|---|---|
| `src/graphloc_core.py` | tokenizer, BM25, graph loader |
| `src/graphloc.py` | the method |
| `src/coverage.py` | coverage-vs-budget main result |
| `src/hop_scale.py` | cap selection |
| `src/hop_profile.py` | hop-distance distribution |
| `src/rerank.py` | negative result 8.1 |
| `src/shortlist.py` | negative result 8.4 |
| `skills/graph-locate/` | ADK skill for agent submissions |

### Citation

```bibtex
@misc{calvi2026graphloc,
  title        = {Where Does the Graph Help? Seed Expansion for
                  Function-Level Bug Localization on Repository Code Graphs},
  author       = {Vitor Calvi},
  year         = {2026},
  note         = {Kaggle: Google -- The Gemma 4 Developer Agent Paper Track},
  howpublished = {\\url{https://www.kaggle.com/competitions/gemma-4-developer-agent-paper}}
}
```

### License

Apache-2.0, matching the competition. No proprietary data, model
checkpoints, or evaluation infrastructure are used. Underlying
benchmarks: SWE-bench (Jimenez et al., ICLR 2024) and its Lite split.